TAMER 베이스라인 모델 불러오기

In [1]:
# Connect Google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# TAMER GitHub repo clone (최초 1회)
%cd "/content/drive/MyDrive/AI_Project"
# !git clone https://github.com/qingzhenduyu/TAMER.git
%cd TAMER

/content/drive/MyDrive/AI_Project
/content/drive/MyDrive/AI_Project/TAMER


In [3]:
# Install requirements
!pip install einops==0.3.0
!pip install editdistance
!pip install pytorch-lightning==1.9.4

# for pytorch-lightning CLI
!pip install jsonargparse[signatures]==3.17.0

# dev-dependency
!pip install flake8==3.9.0
!pip install black==22.3.0
!pip install isort==5.8.0
!pip install jupyter==1.0.0
!pip install opencv-python-headless --no-cache-dir # -headless option
!pip install matplotlib==3.5.1

# for test in crohme
!pip install typer==0.4.1
!pip install beautifulsoup4==4.10.0
!pip install lxml>=4.9.1

  Using cached typer-0.4.1-py3-none-any.whl.metadata (11 kB)
Using cached typer-0.4.1-py3-none-any.whl (27 kB)
  Attempting uninstall: typer
    Found existing installation: typer 0.16.0
    Uninstalling typer-0.16.0:
      Successfully uninstalled typer-0.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.31.0 requires typer<1.0,>=0.12; sys_platform != "emscripten", but you have typer 0.4.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.4
    Uninstalling beautifulsoup4-4.13.4:
      Successfully uninstalled beautifulsoup4-4.13.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfin

In [ ]:
# Check dataset
!ls -R ./data/crohme
print("__________________")
!ls -R ./data/hme100k

./data/crohme:
2014  2016  2019  dictionary.txt  train

./data/crohme/2014:
caption.txt  images.pkl

./data/crohme/2016:
caption.txt  images.pkl

./data/crohme/2019:
caption.txt  images.pkl

./data/crohme/train:
caption.txt  images.pkl
__________________
./data/hme100k:
dictionary.txt	test

./data/hme100k/test:
caption.txt  images.pkl


In [ ]:
# Check total number of CROHME images
import pickle

with open('./data/crohme/2014/images.pkl', 'rb') as f:
    images = pickle.load(f)
print("Total number of 2014 images:", len(images))

with open('./data/crohme/2016/images.pkl', 'rb') as f:
    images = pickle.load(f)
print("Total number of 2016 images:", len(images))

with open('./data/crohme/2019/images.pkl', 'rb') as f:
    images = pickle.load(f)
print("Total number of 2019 images:", len(images))

Total number of 2014 images: 986
Total number of 2016 images: 1147
Total number of 2019 images: 1199


In [ ]:
# Check total number of HME100K images
import pickle

with open('./data/hme100k/test/images.pkl', 'rb') as f:
    images = pickle.load(f)
print("Total number of HME100K images:", len(images))

Total number of HME100K images: 24607


In [ ]:
# Check whether caption.txt. and images.pkl matched in CROHME
import os

# List of dataset folders to check
years = ['2014', '2016', '2019']
base_path = './data/crohme'

for year in years:
    print(f"Checking dataset for year: {year}")

    image_pkl_path = os.path.join(base_path, year, 'images.pkl')
    caption_txt_path = os.path.join(base_path, year, 'caption.txt')

    # Check if files exist
    if not os.path.exists(image_pkl_path):
        print(f"Missing file: {image_pkl_path}")
        continue
    if not os.path.exists(caption_txt_path):
        print(f"Missing file: {caption_txt_path}")
        continue

    # Load data
    with open(image_pkl_path, 'rb') as f:
        images = pickle.load(f)
    with open(caption_txt_path, 'r') as f:
        captions = f.readlines()

    # Check for missing image keys
    missing_images = []
    for line in captions:
        img_name = line.strip().split()[0]
        if img_name not in images:
            missing_images.append(img_name)

    if missing_images:
        print(f"{len(missing_images)} in caption.txt missed")
        print("Example missing keys:", missing_images[:5], "\n")
    else:
        print(f"All {len(captions)} in caption.txt exist\n")

Checking dataset for year: 2014
All 986 in caption.txt exist

Checking dataset for year: 2016
All 1147 in caption.txt exist

Checking dataset for year: 2019
All 1199 in caption.txt exist



In [ ]:
# Check whether caption.txt. and images.pkl matched in HME100K
import os

base_path = './data/hme100k/test'
print(f"Checking dataset for test")

image_pkl_path = os.path.join(base_path, 'images.pkl')
caption_txt_path = os.path.join(base_path, 'caption.txt')

# Check if files exist
if not os.path.exists(image_pkl_path):
    print(f"Missing file: {image_pkl_path}")
if not os.path.exists(caption_txt_path):
    print(f"Missing file: {caption_txt_path}")

# Load data
with open(image_pkl_path, 'rb') as f:
    images = pickle.load(f)
with open(caption_txt_path, 'r') as f:
    captions = f.readlines()

# Check for missing image keys
missing_images = []
for line in captions:
    img_name = line.strip().split()[0]
    if img_name not in images:
        missing_images.append(img_name)

if missing_images:
    print(f"{len(missing_images)} in caption.txt missed")
    print("Example missing keys:", missing_images[:5], "\n")
else:
    print(f"All {len(captions)} in caption.txt exist\n")

Checking dataset for test
All 24607 in caption.txt exist



In [ ]:
# Run test
import os
import torch
import pytorch_lightning as pl
from tamer.datamodule.datamodule import HMEDatamodule
from tamer.lit_tamer import LitTAMER

# Allow unpickle
from torch.serialization import add_safe_globals
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
add_safe_globals([ModelCheckpoint])

def test_model(dataset_name, checkpoint_path, data_root, eval_batch_size=1, num_workers=2):
    # Load dataset
    dm = HMEDatamodule(
        folder=data_root,
        test_folder=dataset_name,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers
    )
    dm.setup("test")

    # Load model
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location="cpu")
    model = LitTAMER.load_from_checkpoint(checkpoint_path, map_location="cpu")

    # Test
    print(f"\nTesting on {data_root}/{dataset_name} dataset...")
    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        logger=False,
        enable_progress_bar=True
    )
    trainer.test(model, datamodule=dm)
    print("-" * 50)

CROHME, HME100K 테스트 결과

In [ ]:
# Test CROHME 2014, 2016, 2019
ckpt_path = "/content/drive/MyDrive/AI_Project/TAMER/lightning_logs/version_0/checkpoints/epoch=315-step=118815-val_ExpRate=0.6113.ckpt"

for year in ["2014", "2016", "2019"]:
    test_model(dataset_name=year, checkpoint_path=ckpt_path, data_root="data/crohme", eval_batch_size=1, num_workers=2)

Load data from: data/crohme
Extract data from: 2014, with data size: 986
total  986 batch data loaded


/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/migration/migration.py:195: PossibleUserWarning: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
  rank_zero_warn(
INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.4.9 to v1.9.4. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file lightning_logs/version_0/checkpoints/epoch=315-step=118815-val_ExpRate=0.6113.ckpt`
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first w


Testing on data/crohme/2014 dataset...
Extract data from: 2014, with data size: 986
total  986 batch data loaded


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: 0it [00:00, ?it/s]

...

Validation ExpRate: 0.5963382720947266
Validation 1-error Rate: 0.767218831735
Validation 2-error Rate: 0.835222319093
--------------------------------------------------
Load data from: data/crohme
Extract data from: 2019, with data size: 1199
total  1199 batch data loaded


INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.4.9 to v1.9.4. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file lightning_logs/version_0/checkpoints/epoch=315-step=118815-val_ExpRate=0.6113.ckpt`
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Testing on data/crohme/2019 dataset...
Extract data from: 2019, with data size: 1199
total  1199 batch data loaded


Testing: 0it [00:00, ?it/s]

Validation ExpRate: 0.6213511228561401
Validation 1-error Rate: 0.782318598832
Validation 2-error Rate: 0.850708924103
--------------------------------------------------


In [ ]:
# Test HME100K
ckpt_path = "/content/drive/MyDrive/AI_Project/TAMER/lightning_logs/version_1/checkpoints/epoch=51-step=162967-val_ExpRate=0.6851.ckpt"

test_model(dataset_name="test", checkpoint_path=ckpt_path, data_root="data/hme100k", eval_batch_size=4, num_workers=2) #num_workers=4 -> 2

Load data from: data/hme100k
Extract data from: test, with data size: 24607
total  7775 batch data loaded


INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.4.9 to v1.9.4. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file lightning_logs/version_1/checkpoints/epoch=51-step=162967-val_ExpRate=0.6851.ckpt`
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs



Testing on data/hme100k/test dataset...
Extract data from: test, with data size: 24607


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


total  7775 batch data loaded


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Testing: 0it [00:00, ?it/s]

Validation ExpRate: 0.6863900423049927
Validation 1-error Rate: 0.848742227821
Validation 2-error Rate: 0.902182305848
--------------------------------------------------
